# Model Validation

All three models predict the raw price in Philippine pesos. The grids below are the ones reported in the paper. `random_state=42` is fixed for Random Forest and Gradient Boosting.

The submitted model is `../models/best_model.joblib`. This notebook loads that file for the held-out check and does not overwrite it.

The grid-search cells and the cross-validation metrics cell refit models in memory only. Run them when you need to reproduce the search or the fold table. They are slow, and they are not required to use the saved model.

In [23]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## Data and chronological split

In [24]:
df = pd.read_csv("../data/processed/wfp_food_prices_phl_processed.csv")

df["date"] = pd.to_datetime(df["date"])

df.head()

,date,admin1,admin2,market,latitude,longitude,category,commodity,unit,priceflag,pricetype,price,year,month,quarter
0,2000-01-15,National Capital region,Metropolitan Manila,Metro Manila,14.60,120.98,cereals and tubers,"Rice (milled, superior)",KG,actual,Retail,20.0,2000,1,1
1,2000-01-15,Region III,Nueva Ecija,Palayan,15.54,121.08,"meat, fish and eggs",Meat (pork),KG,actual,Retail,77.0,2000,1,1
2,2000-01-15,Region VI,Iloilo,Iloilo City,10.70,122.56,"meat, fish and eggs",Meat (pork),KG,actual,Retail,88.2,2000,1,1
3,2000-01-15,Region VI,Iloilo,Iloilo City,10.70,122.56,cereals and tubers,"Rice (regular, milled)",KG,actual,Wholesale,14.7,2000,1,1
4,2000-01-15,Region VI,Iloilo,Iloilo City,10.70,122.56,cereals and tubers,"Rice (regular, milled)",KG,actual,Retail,17.8,2000,1,1


In [25]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter

In [26]:
df = df.sort_values("date").reset_index(drop=True)
df[["date"]].tail()

,date
242207,2026-08-15
242208,2026-08-15
242209,2026-08-15
242210,2026-08-15
242211,2026-08-15


In [27]:
X = df.drop(columns=[
    "price",
    "date"
])

y = df["price"]

In [28]:
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 193769
Testing rows: 48443


## Shared preprocessing and folds

In [29]:
numeric_features = [
    "latitude",
    "longitude",
    "year",
    "month",
    "quarter"
]

categorical_features = [
    "admin1",
    "admin2",
    "market",
    "category",
    "commodity",
    "unit",
    "priceflag",
    "pricetype"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numeric_features
        )
    ]
)

tscv = TimeSeriesSplit(n_splits=5)

## Linear Regression

Linear Regression has no tuned hyperparameters. It is scored on the same five folds through `GridSearchCV` with an empty grid. The fit cell is fast relative to the two ensembles.

In [30]:
lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

lr_cv = GridSearchCV(
    estimator=lr_pipeline,
    param_grid={},
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=2
)

In [33]:
lr_cv.fit(X_train, y_train)

print("Linear Regression CV MAE:", -lr_cv.best_score_)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Linear Regression CV MAE: 29.13160479209361


## Random Forest

The grid compares 50 and 100 trees, a maximum depth of `None` or 20, and `max_features='sqrt'`. Each forest already uses every core, so the folds run one at a time. This fit is the slow cell: expect roughly one to three hours.

In [34]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_param_grid = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [None, 20],
    "model__max_features": ["sqrt"]
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
    verbose=2
)

In [35]:
rf_grid.fit(X_train, y_train)

print("Best Random Forest Parameters:")
print(rf_grid.best_params_)
print("Best CV MAE:")
print(-rf_grid.best_score_)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END model__max_depth=None, model__max_features=sqrt, model__n_estimators=50; total time=  33.8s
[CV] END model__max_depth=None, model__max_features=sqrt, model__n_estimators=50; total time= 1.8min
[CV] END model__max_depth=None, model__max_features=sqrt, model__n_estimators=50; total time= 3.6min
[CV] END model__max_depth=None, model__max_features=sqrt, model__n_estimators=50; total time= 6.6min
[CV] END model__max_depth=None, model__max_features=sqrt, model__n_estimators=50; total time=10.8min
[CV] END model__max_depth=None, model__max_features=sqrt, model__n_estimators=100; total time=  49.4s
[CV] END model__max_depth=None, model__max_features=sqrt, model__n_estimators=100; total time= 3.2min
[CV] END model__max_depth=None, model__max_features=sqrt, model__n_estimators=100; total time= 7.1min
[CV] END model__max_depth=None, model__max_features=sqrt, model__n_estimators=100; total time=13.7min


KeyboardInterrupt: 

## Gradient Boosting

The grid uses 100 estimators, learning rates of 0.05 and 0.10, and tree depths of 3 and 5. Expect roughly 20 to 60 minutes.

In [36]:
gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        )
    ]
)

gb_param_grid = {
    "model__n_estimators": [100],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 5]
}

gb_grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=2
)

In [ ]:
gb_grid.fit(X_train, y_train)

print("Best Gradient Boosting Parameters:")
print(gb_grid.best_params_)
print("Best CV MAE:")
print(-gb_grid.best_score_)

## Cross-validation metrics for the selected settings

The search reported in the paper selected Random Forest with 100 trees, unlimited depth, and `max_features='sqrt'`, and Gradient Boosting with 100 estimators, learning rate 0.10, and maximum depth 5. This cell scores those three settings on the same five training folds and reports MAE, RMSE, and R², including the fold standard deviation.

It refits the three selected models on the training folds only. It does not read the held-out rows and it does not write a model file. Expect roughly one to three hours, mostly from the Random Forest.

In [37]:
selected_models = {
    "Linear Regression": lr_pipeline,
    "Random Forest": clone(rf_pipeline).set_params(
        model__n_estimators=100,
        model__max_depth=None,
        model__max_features="sqrt"
    ),
    "Gradient Boosting": clone(gb_pipeline).set_params(
        model__n_estimators=100,
        model__learning_rate=0.1,
        model__max_depth=5
    ),
}

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

rows = []
for name, model in selected_models.items():
    scores = cross_validate(
        clone(model),
        X_train,
        y_train,
        cv=tscv,
        scoring=scoring,
        n_jobs=1
    )
    for metric in scoring:
        values = scores[f"test_{metric}"]
        if metric != "R2":
            values = -values
        rows.append({
            "Model": name,
            "Metric": metric,
            "Mean": values.mean(),
            "SD": values.std(),
            **{f"Fold {k + 1}": value for k, value in enumerate(values)}
        })

cv_metrics = pd.DataFrame(rows)
cv_metrics.round(3)

,Model,Metric,Mean,SD,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5
0,Linear Regression,MAE,29.132,1.593,26.591,28.807,30.141,31.362,28.757
1,Linear Regression,RMSE,46.143,2.854,42.053,45.045,47.061,50.812,45.743
2,Linear Regression,R2,0.846,0.017,0.822,0.839,0.854,0.842,0.873
3,Random Forest,MAE,23.225,7.479,37.396,24.048,18.931,18.980,16.769
4,Random Forest,RMSE,40.051,11.307,61.246,41.349,32.896,35.503,29.260
5,Random Forest,R2,0.857,0.121,0.622,0.864,0.929,0.923,0.948
6,Gradient Boosting,MAE,26.333,2.578,31.276,26.488,24.467,24.898,24.538
7,Gradient Boosting,RMSE,41.800,3.971,48.868,42.363,38.096,41.725,37.946
8,Gradient Boosting,R2,0.865,0.056,0.759,0.857,0.904,0.893,0.913


## Held-out check of the saved model

This cell only reloads `../models/best_model.joblib` and scores it on the later 20% of rows. It does not fit a model and it cannot overwrite the file.

In [38]:
model_path = Path("../models/best_model.joblib")

if not model_path.exists():
    raise FileNotFoundError(f"Saved model not found: {model_path}")

saved_model = joblib.load(model_path)
saved_predictions = saved_model.predict(X_test)

saved_mae = mean_absolute_error(y_test, saved_predictions)
saved_rmse = np.sqrt(mean_squared_error(y_test, saved_predictions))
saved_r2 = r2_score(y_test, saved_predictions)

print("SAVED RANDOM FOREST TEST RESULTS")
print("---------------------------------")
print(f"MAE:  {saved_mae:.3f}")
print(f"RMSE: {saved_rmse:.3f}")
print(f"R²:   {saved_r2:.4f}")
print(f"File size: {model_path.stat().st_size / 1024 ** 2:.1f} MB")
print("The saved model file was not modified.")

SAVED RANDOM FOREST TEST RESULTS
---------------------------------
MAE:  17.271
RMSE: 27.946
R²:   0.9565
File size: 304.6 MB
The saved model file was not modified.
